# Імпорти та налаштування

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

# Константа для відтворюваності результатів
RANDOM_STATE = 42

# Налаштування графіків
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["figure.dpi"] = 120
sns.set_style("whitegrid")

# Шлях для збереження графіків
FIGURES_DIR = "figures"

print("Імпорти завершено успішно.")

# Завантаження та первинний аналіз даних

In [ ]:
# Завантаження датасету
data = load_breast_cancer()

# Перетворення у DataFrame
df = pd.DataFrame(data.data, columns=data.feature_names)
df["target"] = data.target

# Назви класів
target_names = data.target_names  # ['malignant', 'benign']
print(f"Класи: {target_names}")
print(f"Кількість об'єктів: {df.shape[0]}")
print(f"Кількість ознак: {df.shape[1] - 1}")
print(f"Кількість класів: {len(target_names)}")

In [ ]:
# Перші 5 рядків
df.head()

In [ ]:
# Статистичний опис ознак
df.describe()

In [ ]:
# Інформація про типи даних та пропущені значення
print("Інформація про датасет:")
df.info()
print(f"\nПропущені значення:\n{df.isnull().sum().sum()} (загалом)")

In [ ]:
# Розподіл класів
class_counts = df["target"].value_counts()
print("Розподіл класів:")
for idx, count in class_counts.items():
    print(f"  {target_names[idx]} ({idx}): {count} ({count / len(df) * 100:.1f}%)")

# Візуалізація розподілу класів
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(target_names, class_counts.sort_index().values, color=["#e74c3c", "#2ecc71"])
ax.set_title("Розподіл класів у датасеті")
ax.set_xlabel("Клас")
ax.set_ylabel("Кількість")
for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
            str(int(bar.get_height())), ha="center", fontweight="bold")
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/class_distribution.png", bbox_inches="tight")
plt.show()

# Підготовка даних

In [ ]:
# Розділення на ознаки та цільову змінну
X = df.drop("target", axis=1)
y = df["target"]

# Розділення на тренувальну (80%) та тестову (20%) вибірки
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Розмір тренувальної вибірки: {X_train.shape[0]} ({X_train.shape[0] / len(X) * 100:.0f}%)")
print(f"Розмір тестової вибірки:     {X_test.shape[0]} ({X_test.shape[0] / len(X) * 100:.0f}%)")
print(f"\nРозподіл класів у train: {dict(y_train.value_counts().sort_index())}")
print(f"Розподіл класів у test:  {dict(y_test.value_counts().sort_index())}")

In [ ]:
# Стандартизація ознак
# fit_transform() — тільки на тренувальних даних
# transform() — на тестових (щоб уникнути витоку інформації)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Стандартизацію виконано.")
print(f"Середнє ознак (train): {X_train_scaled.mean(axis=0)[:3].round(6)} ...")
print(f"Std ознак (train):     {X_train_scaled.std(axis=0)[:3].round(6)} ...")

# Побудова baseline-моделей

In [ ]:
# Модель 1: Логістична регресія
lr_model = LogisticRegression(random_state=RANDOM_STATE, max_iter=10000)
lr_model.fit(X_train_scaled, y_train)

# Прогнози
y_pred_lr_train = lr_model.predict(X_train_scaled)
y_pred_lr_test = lr_model.predict(X_test_scaled)

print("Logistic Regression навчено.")
print(f"  Accuracy (train): {accuracy_score(y_train, y_pred_lr_train):.4f}")
print(f"  Accuracy (test):  {accuracy_score(y_test, y_pred_lr_test):.4f}")

In [ ]:
# Модель 2: Випадковий ліс (Random Forest)
rf_model = RandomForestClassifier(random_state=RANDOM_STATE)
rf_model.fit(X_train_scaled, y_train)

# Прогнози
y_pred_rf_train = rf_model.predict(X_train_scaled)
y_pred_rf_test = rf_model.predict(X_test_scaled)

print("Random Forest навчено.")
print(f"  Accuracy (train): {accuracy_score(y_train, y_pred_rf_train):.4f}")
print(f"  Accuracy (test):  {accuracy_score(y_test, y_pred_rf_test):.4f}")

# Оцінка якості моделей

In [ ]:
# Classification Report — Logistic Regression
print("=" * 60)
print("LOGISTIC REGRESSION — Classification Report")
print("=" * 60)
print(classification_report(y_test, y_pred_lr_test, target_names=target_names))

In [ ]:
# Classification Report — Random Forest
print("=" * 60)
print("RANDOM FOREST — Classification Report")
print("=" * 60)
print(classification_report(y_test, y_pred_rf_test, target_names=target_names))

In [ ]:
# Confusion Matrix — візуалізація для обох моделей
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

models_data = [
    ("Logistic Regression", y_pred_lr_test, "confusion_matrix_lr.png"),
    ("Random Forest", y_pred_rf_test, "confusion_matrix_rf.png"),
]

for ax, (name, y_pred, filename) in zip(axes, models_data):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=target_names,
                yticklabels=target_names, ax=ax)
    ax.set_title(f"Confusion Matrix — {name}")
    ax.set_xlabel("Передбачений клас")
    ax.set_ylabel("Справжній клас")

plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/confusion_matrix_lr.png", bbox_inches="tight")
plt.show()

# Збереження окремих confusion matrix
for name, y_pred, filename in models_data:
    fig, ax = plt.subplots(figsize=(5, 4))
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=target_names,
                yticklabels=target_names, ax=ax)
    ax.set_title(f"Confusion Matrix — {name}")
    ax.set_xlabel("Передбачений клас")
    ax.set_ylabel("Справжній клас")
    plt.tight_layout()
    plt.savefig(f"{FIGURES_DIR}/{filename}", bbox_inches="tight")
    plt.close()

In [ ]:
# Зведена таблиця метрик
metrics_data = []
for name, y_pred_train, y_pred_test in [
    ("Logistic Regression", y_pred_lr_train, y_pred_lr_test),
    ("Random Forest", y_pred_rf_train, y_pred_rf_test),
]:
    metrics_data.append({
        "Модель": name,
        "Accuracy (train)": accuracy_score(y_train, y_pred_train),
        "Accuracy (test)": accuracy_score(y_test, y_pred_test),
        "Precision": precision_score(y_test, y_pred_test, average="weighted"),
        "Recall": recall_score(y_test, y_pred_test, average="weighted"),
        "F1-score": f1_score(y_test, y_pred_test, average="weighted"),
    })

metrics_df = pd.DataFrame(metrics_data).set_index("Модель")
metrics_df = metrics_df.round(10)
print("Зведена таблиця метрик:")
metrics_df

In [ ]:
# Візуалізація порівняння метрик
fig, ax = plt.subplots(figsize=(10, 5))
metrics_plot = metrics_df[["Accuracy (test)", "Precision", "Recall", "F1-score"]]
metrics_plot.plot(kind="bar", ax=ax, rot=0, colormap="Set2")
ax.set_title("Порівняння метрик моделей")
ax.set_ylabel("Значення")
ax.set_ylim(0.9, 1.0)
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/metrics_comparison.png", bbox_inches="tight")
plt.show()

# Аналіз результатів

In [ ]:
# Перевірка overfitting: порівняння accuracy на train vs test
print("Перевірка перенавчання (overfitting):")
print(f"{'Модель':<25} {'Train Acc':>10} {'Test Acc':>10} {'Різниця':>10}")
print("-" * 55)
for name, y_pred_train, y_pred_test in [
    ("Logistic Regression", y_pred_lr_train, y_pred_lr_test),
    ("Random Forest", y_pred_rf_train, y_pred_rf_test),
]:
    train_acc = accuracy_score(y_train, y_pred_train)
    test_acc = accuracy_score(y_test, y_pred_test)
    diff = train_acc - test_acc
    print(f"{name:<25} {train_acc:>10.4f} {test_acc:>10.4f} {diff:>10.4f}")

print("\n* Якщо різниця між train та test accuracy значна (>0.05),")
print("  це може свідчити про перенавчання моделі.")

In [ ]:
# Найважливіші ознаки (Feature Importance) — Random Forest
importances = rf_model.feature_importances_
feature_importance_df = pd.DataFrame({
    "Ознака": data.feature_names,
    "Важливість": importances,
}).sort_values("Важливість", ascending=False)

# Топ-10 найважливіших ознак
top_n = 10
fig, ax = plt.subplots(figsize=(10, 6))
top_features = feature_importance_df.head(top_n)
ax.barh(top_features["Ознака"][::-1], top_features["Важливість"][::-1], color="#3498db")
ax.set_title(f"Топ-{top_n} найважливіших ознак (Random Forest)")
ax.set_xlabel("Важливість")
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/feature_importance.png", bbox_inches="tight")
plt.show()

print(f"\nТоп-{top_n} ознак:")
for i, row in top_features.iterrows():
    print(f"  {row['Ознака']}: {row['Важливість']:.4f}")

# Додаткове завдання: крос-валідація та дослідження гіперпараметрів

In [ ]:
# k-fold крос-валідація (k=5)
k = 5
print(f"Крос-валідація (k={k}):")
print("-" * 50)

for name, model in [
    ("Logistic Regression", LogisticRegression(random_state=RANDOM_STATE, max_iter=10000)),
    ("Random Forest", RandomForestClassifier(random_state=RANDOM_STATE)),
]:
    scores = cross_val_score(model, X_train_scaled, y_train, cv=k, scoring="accuracy")
    print(f"{name}:")
    print(f"  Scores: {scores.round(4)}")
    print(f"  Mean:   {scores.mean():.4f} (+/- {scores.std():.4f})")
    print()

In [ ]:
# Дослідження впливу гіперпараметра C на Logistic Regression
C_values = np.logspace(-3, 2, 20)  # від 0.001 до 100
train_scores = []
test_scores = []

for C in C_values:
    model = LogisticRegression(C=C, random_state=RANDOM_STATE, max_iter=10000)
    model.fit(X_train_scaled, y_train)
    train_scores.append(accuracy_score(y_train, model.predict(X_train_scaled)))
    test_scores.append(accuracy_score(y_test, model.predict(X_test_scaled)))

# Графік залежності accuracy від C
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(C_values, train_scores, "o-", label="Train Accuracy", color="#2ecc71")
ax.plot(C_values, test_scores, "s-", label="Test Accuracy", color="#e74c3c")
ax.set_xscale("log")
ax.set_xlabel("Параметр C (log scale)")
ax.set_ylabel("Accuracy")
ax.set_title("Вплив параметра C на якість Logistic Regression")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/hyperparameter_c.png", bbox_inches="tight")
plt.show()

# Оптимальне значення C
best_idx = np.argmax(test_scores)
print(f"Оптимальне значення C: {C_values[best_idx]:.4f}")
print(f"  Train Accuracy: {train_scores[best_idx]:.4f}")
print(f"  Test Accuracy:  {test_scores[best_idx]:.4f}")